# 🧹 Cuaderno 2: Pipeline de Limpieza, Transformación e Ingeniería de Características
## Proyecto de Ciencia de Datos (X Semestre) - Segundo Corte
### Caso de Estudio: Inside Airbnb (New York City)

---

### 🎯 Objetivos de este Cuaderno:
1. **Pipeline de Depuración:** Implementar el proceso sistemático de limpieza justificado en la auditoría de calidad.
2. **Imputación Semántica:** Resolver nulos en variables de infraestructura sin distorsionar la distribución.
3. **Normalización Categórica:** Limpiar textos, homogeneizar categorías y convertir flags booleanos de texto ('t'/'f' a 1/0).
4. **Desanidado de Atributos:** Extraer variables binarias para las comodidades (*amenities*) clave.
5. **Feature Engineering:** Calcular distancias geodésicas (Haversine) a puntos turísticos de NYC y ratios de capacidad.
6. **Entregables:** Exportar el dataset limpio y enriquecido en formatos **CSV**, **Parquet** y **Excel** (`.xlsx`).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("../..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_raw_listings
from src.cleaning import (
    clean_airbnb_dataset,
    drop_empty_rows,
    filter_incomplete_rows,
    remove_duplicates,
    normalize_categorical_features
)
from src.features import engineer_features

print("✅ Módulos de limpieza e ingeniería importados correctamente.")

✅ Módulos de limpieza e ingeniería importados correctamente.


---
## 1. Carga del Dataset Crudo y Depuración Inicial

In [2]:
df_raw = load_raw_listings("nyc")
print(f"Dimensiones iniciales: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas.")

# Paso 1: Purgado de filas vacías e incompletas
df_step1 = drop_empty_rows(df_raw)
df_step1 = filter_incomplete_rows(df_step1, min_valid_cols=15)

# Paso 2: Deduplicación
df_step2 = remove_duplicates(df_step1, subset=["id"])
print(f"Dimensiones tras filtros de integridad: {df_step2.shape[0]:,} filas.")

📖 Cargando /Users/jyarar/projects/u/x-semestre/data-science/data/selected/airbnb/raw/nyc/listings.csv.gz...


📊 Dataset cargado: 30,259 filas x 90 columnas.
Dimensiones iniciales: 30,259 filas x 90 columnas.
✅ No se detectaron duplicados para el criterio especificado.
Dimensiones tras filtros de integridad: 30,259 filas.


---
## 2. Ejecución del Pipeline Modular de Limpieza

Aplicamos las transformaciones de:
- Parseo de monedas (`price` a `price_usd`).
- Parseo de baños (`bathrooms_text` a `bathrooms_num`).
- Poda de outliers justificada (\$10 ≤ precio ≤ \$2,000 USD; estancia mínima ≤ 365 días).
- Imputación semántica de `bedrooms_num` y `beds_num` según capacidad.
- Binarización de 26 comodidades clave.

In [3]:
df_clean = clean_airbnb_dataset(
    df_step2,
    min_price=10.0,
    max_price=2000.0,
    max_min_nights=365
)

# Normalización de textos y booleanos
df_clean = normalize_categorical_features(df_clean)

print(f"Dimensiones tras pipeline de limpieza: {df_clean.shape[0]:,} filas x {df_clean.shape[1]} columnas.")
print(f"Nulos en price_usd: {df_clean['price_usd'].isna().sum()}")
print(f"Rango de precios: [${df_clean['price_usd'].min():.2f}, ${df_clean['price_usd'].max():.2f}] USD")

Dimensiones tras pipeline de limpieza: 21,347 filas x 122 columnas.
Nulos en price_usd: 0
Rango de precios: [$10.23, $1999.25] USD


---
## 3. Ingeniería de Características (Feature Engineering)

Incorporamos variables derivadas con alto valor explicativo y predictivo:
1. **Distancias Geodésicas (Fórmula de Haversine):**
   - Distancia a Times Square (km)
   - Distancia a Central Park (km)
   - Distancia a Financial District (km)
   - Distancia a Brooklyn Bridge (km)
2. **Ratios de Capacidad e Infraestructura:**
   - Huéspedes por habitación (`accommodates_per_bedroom`)
   - Camas por habitación (`beds_per_bedroom`)
   - Baños por huésped (`bathrooms_per_accommodates`)
3. **Métricas de Reseñas:** Indicador binario de historial previo (`has_reviews`) y puntuación limpia.

In [4]:
df_features = engineer_features(df_clean)
print(f"Dimensiones finales enriquecidas: {df_features.shape[0]:,} filas x {df_features.shape[1]} columnas.")

# Muestra de las nuevas columnas geoespaciales
geo_cols = [c for c in df_features.columns if "dist_" in c]
print("\nVariables de distancia calculadas:")
display(df_features[geo_cols].describe().round(2))

Dimensiones finales enriquecidas: 21,347 filas x 136 columnas.

Variables de distancia calculadas:


,dist_times_square_km,dist_financial_district_km,dist_central_park_km,dist_brooklyn_bridge_km,min_dist_to_landmark_km
count,21347.00,21347.00,21347.00,21347.00,21347.00
mean,7.87,8.87,9.02,8.37,5.60
std,5.64,5.22,5.78,5.13,5.02
min,0.01,0.02,0.15,0.35,0.01
25%,3.35,5.29,4.13,4.75,1.74
50%,7.22,7.69,7.98,6.94,4.21
75%,10.99,11.45,12.53,10.94,7.74
max,35.92,30.35,39.19,30.93,30.35


---
## 4. Exportación y Validación de los Entregables

Guardamos el dataset limpio para su uso directo en el Segundo y Tercer Corte en tres formatos:
- `airbnb_nyc_limpio.parquet` (para alta eficiencia en modelos de Machine Learning).
- `airbnb_nyc_limpio.csv` (formato universal interoperable).
- `airbnb_nyc_limpio.xlsx` (entregable en Excel para inspección analítica).

In [5]:
output_dir = PROJECT_ROOT / "data" / "selected" / "airbnb" / "processed" / "segundo_corte"
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "airbnb_nyc_limpio.csv"
parquet_path = output_dir / "airbnb_nyc_limpio.parquet"
xlsx_path = output_dir / "airbnb_nyc_limpio.xlsx"

# Guardar CSV
df_features.to_csv(csv_path, index=False)
print(f"✅ CSV guardado: {csv_path.name} ({csv_path.stat().st_size / (1024*1024):.1f} MB)")

# Guardar Parquet
df_features.to_parquet(parquet_path, index=False)
print(f"✅ Parquet guardado: {parquet_path.name} ({parquet_path.stat().st_size / (1024*1024):.1f} MB)")

# Guardar Excel
df_excel = df_features.copy()
for col in df_excel.select_dtypes(include=['object']).columns:
    df_excel[col] = df_excel[col].astype(str)
df_excel.to_excel(xlsx_path, index=False, engine="openpyxl")
print(f"✅ Excel guardado: {xlsx_path.name} ({xlsx_path.stat().st_size / (1024*1024):.1f} MB)")

✅ CSV guardado: airbnb_nyc_limpio.csv (63.9 MB)
✅ Parquet guardado: airbnb_nyc_limpio.parquet (15.0 MB)


/var/folders/wc/4_xf4hl57tl69mp_n62ffl840000gn/T/ipykernel_86306/3577547175.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_excel.select_dtypes(include=['object']).columns:


✅ Excel guardado: airbnb_nyc_limpio.xlsx (25.1 MB)


---
## 5. Tabla Resumen Comparativa: Antes vs. Después de Limpieza

| Métrica / Dimensión | Dataset Crudo | Dataset Limpio (Segundo Corte) | Impacto / Justificación |
| :--- | :--- | :--- | :--- |
| **Número de Filas** | 30,259 | 21,347 | Poda controlada de listados sin precio y outliers extremos. |
| **Número de Columnas** | 90 | 136 | +46 características nuevas (26 amenities + distancias + ratios). |
| **Nulos en Variable Target (`price`)** | 8,744 (28.9%) | **0 (0.0%)** | Dataset 100% supervisado listo para modelado. |
| **Tipo de Dato de Precio** | String (`"$150.00"`) | Float (`price_usd`) + Float (`log_price`) | Normalizado para regresión lineal y de gradiente. |
| **Rango de Precios** | \$0 a \$10,000+ USD | \$10.23 a \$1,999.25 USD | Eliminación de ruido y publicaciones ficticias. |
| **Infraestructura (`bedrooms`, `beds`)** | ~33% Nulos | **0 Nulos** | Imputación semántica coherente con `accommodates`. |